In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from google.colab import files
files = files.upload()
NVDA = pd.read_csv("NVDA.csv")
NVDA['Date'] = pd.to_datetime(NVDA['Date'])

FileNotFoundError: [Errno 2] No such file or directory: 'nvda_us_d (1).csv'

In [ ]:
NVDA['MA_5'] = NVDA['Close'].rolling(5).mean()
NVDA['MA_10'] = NVDA['Close'].rolling(10).mean()
NVDA['MA_20'] = NVDA['Close'].rolling(20).mean()
NVDA['Close_t-1'] = NVDA['Close'].shift(1)
NVDA['Close_t-2'] = NVDA['Close'].shift(2)
NVDA['Close_t-3'] = NVDA['Close'].shift(3)
NVDA['Close_t-4'] = NVDA['Close'].shift(4)
NVDA['Close_t-5'] = NVDA['Close'].shift(5)

In [ ]:
delta = NVDA['Close'].diff()

gains = delta.clip(lower=0)
losses = -delta.clip(upper=0)

avg_gains = gains.rolling(14).mean()
avg_losses = losses.rolling(14).mean()

RS = (avg_gains / avg_losses)
NVDA["RSI_14"] = 100 - (100 / (1 + RS))

In [ ]:
EMA_12 = NVDA['Close'].ewm(span=12,adjust=False).mean()
EMA_26 = NVDA['Close'].ewm(span=26,adjust=False).mean()
NVDA['MACD'] = EMA_12-EMA_26
NVDA['MACD_Signal'] = NVDA['MACD'].ewm(span=9,adjust=False).mean()
NVDA['MACD_Histogram'] = NVDA['MACD_Signal'] - NVDA['MACD']
NVDA.reset_index(drop=True, inplace=True)
NVDA.ffill(inplace=True)
NVDA.bfill(inplace=True)

In [ ]:
NVDA['Next_Day_Close'] = NVDA['Close'].shift(-1)
NVDA = NVDA.dropna(subset=['Next_Day_Close'])

In [ ]:
NVDA['Return'] = NVDA['Close'].pct_change()*100
NVDA['Target'] = NVDA['Return'].shift(-1)
NVDA = NVDA.dropna()

In [ ]:
window_size = 20
X = []
y = []
features = ['Close','Open','High','Low','Volume','MA_5','MA_10','MA_20','RSI_14','MACD']
for i in range(len(NVDA)-window_size):
  X_window = NVDA[features].iloc[i:i+window_size].values.flatten()
  y_target = NVDA['Target'].iloc[i+window_size]

  X.append(X_window)
  y.append(y_target)

X = np.array(X)
y = np.array(y)

In [ ]:
total_samples = len(X)
split_indx = int(total_samples * 0.8)

X_train = X[:split_indx]
y_train = y[:split_indx]


X_test = X[split_indx:]
y_test = y[split_indx:]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
MinMax = MinMaxScaler(feature_range=(0,1))
X_train = MinMax.fit_transform(X_train)
X_test = MinMax.transform(X_test)

In [ ]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate = 0.05,
    subsample = 0.9,
    colsample_bytree = 0.7,
    random_state = 0
    )
xgb.fit(X_train,y_train)
y_pred_test = xgb.predict(X_test)
y_pred_train = xgb.predict(X_train)
from sklearn.metrics import r2_score
r2_test = r2_score(y_test,y_pred_test)
r2_train = r2_score(y_train,y_pred_train)
print("R2 test:",r2_test)
print("R2 train:",r2_train)

R2 test: -0.11746304501070193
R2 train: 0.3752520372003797


In [ ]:
NVDA['Date'] = pd.to_datetime(NVDA['Date'])
dates = NVDA['Date'].iloc[-len(y_test):]
graph_of_model = px.scatter(x=dates,y=y_test)
graph_of_model.add_scatter(x=dates,y=y_pred_test,mode='lines',name='Prediction Line')
graph_of_model.show()

In [ ]:
last_window = NVDA[features].iloc[-window_size:].values.flatten()
X_next_day = np.array(last_window).reshape(1,-1)
next_day_pred = xgb.predict(X_next_day)
next_day_pred = next_day_pred[0]
if next_day_pred>0:
  print('Predicted next day change in price is an increase by', round(next_day_pred,3),'%')
elif next_day_pred == 0.00:
  print('Predicted next day change in price is by 0.00%')
else:
  print('Predicted next day change in price is a decrease by ', round(next_day_pred,3),'%')

Predicted next day change in price is a decrease by  -0.297 %
